## Imports

In [ ]:
import os
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import plotly
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import statsmodels.api as sm
import scipy.stats as stats
from scipy.stats import spearmanr
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association
from statsmodels.graphics.api import abline_plot
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn import linear_model, preprocessing 
from statsmodels.nonparametric.smoothers_lowess import lowess
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.filterwarnings(action="ignore", module="scipy", message="^internal gelsd")

In [ ]:
DATA_DIR = '/Users/chrischoi/Desktop/springboard/nhl-shots/data/clean'
os.listdir(DATA_DIR)
csv = os.path.join(DATA_DIR, 'mp_shots_clean.csv')
df = pd.read_csv(csv)

In [ ]:
df.shape

#### When determining the likelihood of any given shot being a goal, for this exercise, I will analyze agnostic of the team and player.

In [ ]:
for col in df.columns.sort_values():
    print(col)

In [ ]:
vars_to_plot = ['averagerestdifference',
                'distancefromlastevent', 
                'homeskatersonice', 'awayskatersonice', 'lasteventshotangle',
                'lasteventxcord_adjusted', 'lasteventycord_adjusted', 
                'shootingteamforwardsonice', 'shootingteammaxtimeonice',
                'shotangleadjusted', 'shotangleplusrebound', 'shotangleplusreboundspeed', 
                'shotdistance', 'speedfromlastevent', 'time', 'timesincefaceoff', 'xcord', 'ycord']

In [ ]:
# Drop columns
# Redundant instances of measurements such as xcord, ycord, and shot distance that are captured in other variables.

columns_to_drop = ['game_id', 'teamcode', 'hometeamcode','awayteamcode', 'shooterplayerid', 'goalieidforshot', 
                    'playernumthatdidlastevent', 'playernumthatdidevent', 'shotid', 'id', 'gameid','game_key', 'date', 'shootername', 'goalienameforshot', 
    
    # Predictive
                    'xgoal', 'xfroze', 'xrebound','xplaycontinuedinzone', 'xplaycontinuedoutsidezone', 'xplaystopped', 'xshotwasongoal',
    
    # Redundant
                    'arenaadjustedxcord', 'arenaadjustedycord', 'arenaadjustedycordabs', 'arenaadjustedshotdistance',
                    'arenaadjustedxcordabs', 'isplayoffgame', 
    
    # Variables reliant on outcome of shot
                    'shotgoaliefroze', 'shotplaycontinuedinzone', 'shotplaycontinuedoutsidezone', 'shotplaystopped', 'shotgeneratedrebound', 'event'
                  ]
df = df.drop(columns = columns_to_drop, errors='ignore')
for col in df.columns.sort_values():
    print(col)

In [ ]:
# Time on ice variables have a lot of outliers that need to be dropped.
# Missing max/average values have been set to 0.0, min to 999.0
for col in df.columns:
    if 'timeonice' in col:
        if 'max' in col:
            df[col] = df[col].replace(0.0, np.nan)
        elif 'min' in col:
            df[col] = df[col].replace(999.0, np.nan)
        elif 'average' in col:
            df[col] = df[col].replace(0.0, np.nan)
            df[col] = df[col].replace(999.0, np.nan)

In [ ]:
int_shots = df.select_dtypes('int64').columns
float_shots = df.select_dtypes('float64').columns

df[int_shots] = df[int_shots].apply(pd.to_numeric, downcast='integer')
df[float_shots] = df[float_shots].apply(pd.to_numeric, downcast='float')

In [ ]:
cols = df.columns
chunk_size = 15

for i in range(0, len(cols), chunk_size):
    display(df[cols[i:i+chunk_size]].describe().T)

In [ ]:
df.groupby('goal')[['period', 'xcordadjusted', 'ycordadjusted', 'shotangleadjusted']].describe().T

In [ ]:
# VERY slow, using large sample to increase performance

df_sample = df.sample(n=100000, random_state = 12)

numeric_cols = df.select_dtypes(include=['number']).columns
corr_df = df_sample[numeric_cols].corr(method='spearman')

In [ ]:
fig = px.imshow(corr_df, height = 1000, width = 1000, color_continuous_scale='plotly')
fig.show()

In [ ]:
corr_pairs = corr_df.unstack()
corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) != corr_pairs.index.get_level_values(1)]
corr_pairs = corr_pairs.drop_duplicates()
corr_pairs_sorted = corr_pairs.sort_values(ascending=False)
corr_pairs_sorted.head(20)

In [ ]:
corr_pairs_filtered = corr_pairs[~(corr_pairs.index.get_level_values(0).str.contains('timeonice')|
                                   corr_pairs.index.get_level_values(1).str.contains('timeonice'))]
corr_pairs_fs = corr_pairs_filtered.sort_values(ascending=False)
corr_pairs_fs.head(20)

In [ ]:
goal_corr = corr_df['goal'].sort_values(key=lambda x: x.abs(), ascending=False)
goal_corr[1:16]      # skips 0 because 0 is goal

The above variables are the fifteen numerical variables that have the greatest correlation with goal percentage. Most of these make sense, such as shot distance negatively impacting the scoring chances of a shot (the further away the shot, the less likely it is that shot is a goal). A few of these require added context in order to fully understand how they impact the scoring percentage of a shot, such as timeuntilnextevent. Specifically, variables like homepenalty1timeleft will only make sense with added context, such as whether it is the home or away team taking the shot. This will require conditional analysis of these variables to understand their impact. In this case: Is the team that took the shot the home or away team? Just based on what I see here, the values that seem most promising are shot location (xcord, ycord, angle, distance) and rebounds. While shot on empty net is going to be important when predicting if a shot is a goal, a very small percentage of the total shots are on empty nets. In most cases, this will add little to predicting the outcome of any given shot.

In [ ]:
fig = px.density_heatmap(df,
                         x='xcord',
                         y='ycord',
                         width=1000, height=575,
                         title= 'All Shot Locations',
                         color_continuous_scale='dense'
                        )
fig.show()

### Binning angle and distance

In [ ]:
df_binned = df.copy()
df_binned['distance_bin'] = pd.cut(df_binned['shotdistance'], bins=200)
df_binned['angle_bin'] = pd.cut(df_binned['shotangleadjusted'], bins=200)

In [ ]:
heatmap_df = (
    df_binned
    .groupby(['distance_bin', 'angle_bin'])
    .agg(
        shots=('goal', 'count'),
        goals=('goal', 'sum')
    )
    .reset_index()
)
heatmap_df['goal_pct'] = heatmap_df['goals'] / heatmap_df['shots']

In [ ]:
heatmap_df['distance_bin'] = heatmap_df['distance_bin'].apply(lambda x: x.mid)
heatmap_df['angle_bin'] = heatmap_df['angle_bin'].apply(lambda x: x.mid)

In [ ]:
fig = px.density_heatmap(heatmap_df,
                         x='distance_bin',
                         y='angle_bin',
                         z='goal_pct',
                         histfunc='avg',
                         width=1100, height=550,
                         nbinsx = 51, nbinsy = 21,
                         title='Goal Probability by Shot Distance and Angle',
                         color_continuous_scale='dense')
fig.show()

In [ ]:
fig = px.density_heatmap(df,
                         x='shotdistance',
                         y='shotangleadjusted',
                         z='goal',
                         histfunc='avg',
                         width=1100, height=550,
                         nbinsx=50, nbinsy=20,
                         title='Goal Probability by Shot Distance and Angle',
                         color_continuous_scale='dense'
                         )
fig.show()

In [ ]:
def goal_probability_curve_plt(df, x_col, frac=0.2):
    d = df[[x_col, 'goal']].copy()
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    
    if d[x_col].nunique() < 5:
        return
    
    x = d[x_col].values
    y = d['goal'].values
    
    smoothed = lowess(y, x, frac=frac, return_sorted=True)
    
    plt.figure(figsize=(3, 2))
    plt.scatter(x, y, s=5, alpha=0.03)
    plt.plot(smoothed[:, 0], smoothed[:, 1], linewidth=2)
    plt.ylim(0, 1)
    plt.xlabel(x_col)
    plt.ylabel('P(goal)')
    plt.title(f'Goal Probability vs {x_col}')
    plt.show()

In [ ]:
#for i in vars_to_plot:
#    goal_probability_curve(df, i)

### Numeric Correlation

In [ ]:
# Function to bin given columns

def bin_column(df, col, bins=100):
    # Bins col in df, default is 100 bins
    # New column is appended with _bin. 
    # Ex: shotangle_bin if col = shotangle
    
    bin_col = f'{col}_bin'
    df_binned = df.copy()
    df_binned[bin_col] = pd.cut(df_binned[col], bins=bins)
    return df_binned


In [ ]:
# Function to find goal % of binned columns

def scoring_pct(df, binned_col):
    # Input data frame taht has already passed through bin_column
    # Specify the column that has been binned
    # Calculates the scoring% based on binned_col
    
    df_pct = (
        df_binned
        .groupby(binned_col)
        .agg(
            shots=('goal', 'count'),
            goals=('goal', 'sum')
        )
        .reset_index()
    )
    df_pct = df_pct[df_pct['shots'] >= 50]
    df_pct['goal_pct'] = df_pct['goals'] / df_pct['shots']
    df_pct['bin_mid'] = df_pct[binned_col].apply(lambda x: x.mid)
    return df_pct

In [ ]:
np.sort(df['shootingteamaveragetimeonicesincefaceoff'].unique())[0]

In [ ]:
# Plotting function

def bin_plots(df, binned_col, y='goal_pct'):
    # Creates scatter plots for binned data
    df['bin_mid'] = pd.to_numeric(df['bin_mid'], errors='coerce')
    df[y] = pd.to_numeric(df[y], errors='coerce')
    fig = px.scatter(df,
                     x='bin_mid', y=y,
                     title = f'Scoring percentage given {binned_col[:-4]}',
                     width=700, height=550,
                     labels = dict(bin_mid=binned_col[:-4], goal_pct='Scoring Percentage'),
                     trendline='lowess'
                    )
    fig.update_yaxes(range=[0, df['goal_pct'].max() + (df['goal_pct'].max()/10)])
    return fig
               

In [ ]:
numeric_columns = df.select_dtypes(include='number')
num_col = [col for col in numeric_columns.columns if df[col].nunique() >= 11]
num_col.append('goal')
numeric_df = df[num_col]

for col in num_col:

    df_binned = bin_column(df, col)
    binned_col = f'{col}_bin'
    
    shot_pct = scoring_pct(df_binned, binned_col)
    fig = bin_plots(shot_pct, binned_col)
    fig.show()

In [ ]:
# Time on ice variables have a lot of outliers that need to be dropped.
# Missing max/average values have been set to 0.0, min to 999.0
for col in df.columns:
    if 'timeonice' in col:
        if 'max' in col:
            df[col] = df[col].replace(0.0, np.nan)
        elif 'min' in col:
            df[col] = df[col].replace(999.0, np.nan)
        elif 'average' in col:
            df[col] = df[col].replace(0.0, np.nan)
            df[col] = df[col].replace(999.0, np.nan)
# df[df.eq(999.0).any(axis=1)]

In [ ]:
# Rerun plotting
numeric_columns = df.select_dtypes(include='number')
num_col = [col for col in numeric_columns.columns if df[col].nunique() >= 11]
num_col.append('goal')
numeric_df = df[num_col]

for col in num_col:

    df_binned = bin_column(df, col)
    binned_col = f'{col}_bin'
    
    shot_pct = scoring_pct(df_binned, binned_col)
    fig = bin_plots(shot_pct, binned_col)
    fig.show()

The most interesting one here to me is xcordadjusted, as it appears that scoring chances are high around the center of the ice, decrease and bottom out at 40, and peak around 80. This is reflected once again with xcord unadjusted. It means that the shot percentage near the center of the ice (\~40 feet from the goal) is higher than the shot percentage near the blue line (\~20 feet from the goal). 

Most of the variables associated with time on ice seem to trend pretty linearly, with defensive teams who spend longer on the ice more likely to allow high quality shots. Conversely, scoring changes tend to increase the longer the shooting team is on the ice as well, which is sort of the opposite of what I expected given the trend of defensive time. I would be interested to see how time on ice is affected by penalties, as players tend to stay on the ice longer during power plays and penalty kills.

### Categorical correlation

In [ ]:
cat_list = df #.select_dtypes(include=['number', 'category', 'string'])
cat_list = [col for col in cat_list.columns if ((df[col].nunique() <= 10))]
cat_df = df[cat_list]
for col in cat_df.columns:
    cat_df = cat_df[cat_df[col].map(cat_df[col].value_counts()) >= 100]

In [ ]:
#for col in cat_df:
#   print('\n', cat_df.groupby(col)['goal'].mean().sort_values(ascending=False)*100)


In [ ]:
# Realized here that I had inadvertently removed all empty net shots when removing missing goalies
df['shotonemptynet'].nunique()

In [ ]:
cat_corr = []
for col in cat_df:
    if col == 'goal':
        continue
    table = pd.crosstab(cat_df[col], cat_df['goal'])
    chi2, p, dof, expected = stats.chi2_contingency(table)
    cramers = association(table)
    cat_corr.append({
        'column': col,
        'chi2': chi2,
        'p_value': p,
        'dof': dof,
        'cramers_v': cramers,
        'num_categories': table.shape[0],
        'expected_min': expected.min()
    })
    print(f'{(20 - len(col)//2) * '-'} {col} {(20 - len(col)//2) * '-'}  \n\nChi Squared: {chi2}\n\
P-Value: {p:.5f}\nDegrees of Freedom: {dof} \nCramers V:{cramers}\nNumber of Categories: {table.shape[0]}\nExpected Min: {expected.min()}\n\n')

In [ ]:
cat_corr_df = pd.DataFrame(cat_corr)
significant_columns = cat_corr_df[(cat_corr_df['cramers_v'] >= 0.04) & (cat_corr_df['p_value'] <= 0.05)]
significant_columns

Here, the most notable significant variables are shotrebound, shottype, playerpositionthatdidevent, homeskatersonice, and shotongoal. defendingteamforwardsonice and awayskatersonice will be interesting to look into as well, but most of these will prove more interesting in multivariate analysis. With this data, most of the Cramers V values are relatively low, however, given the type of data I am working with, lower values can still be considered relevant. 